# Data Analysis Cycle

##  Question & Goal
**Core Objective:** Solve the annual wheat deficit problem in Pakistan.

### Research Questions:
1. Based on Pakistan's historical population growth and consumption patterns, what is the current annual wheat deficit?
2. What strategic vertical yield targets ($kg/ha$) are required to achieve national wheat self-sufficiency by the year 2030 without expanding physical farmland?


In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Agri_data=pd.read_csv('/kaggle/input/datasets/azharalisoomro/pakistan-agricultural-20002024-faostat-data/FAOSTAT_data_en_6-17-2026.csv')
Popu_data=pd.read_csv('/kaggle/input/datasets/azharalisoomro/world-population-dataset/API_SP.POP.TOTL_DS2_en_csv_v2_406129.csv')


## STEP 1 : Filter and Prepare the Population Data

In [2]:
# Filter specifically for Pakistan
pak_pop = Popu_data[Popu_data['Country Name'] == 'Pakistan']

# Select only the year columns we care about (2000 to 2024)
year_columns = [str(year) for year in range(2000, 2025)]
pak_pop_years = pak_pop[year_columns]
#print(pak_pop_years)

# Melt the horizontal years into a vertical layout (Year and Population columns)
pak_pop_vertical = pak_pop_years.melt(var_name='Year', value_name='Population')

# Convert 'Year' column to integer so it matches the agriculture dataset type
pak_pop_vertical['Year'] = pak_pop_vertical['Year'].astype(int)
print(pak_pop_vertical)

    Year   Population
0   2000  154879127.0
1   2001  159270907.0
2   2002  163222549.0
3   2003  167110248.0
4   2004  171286000.0
5   2005  175453212.0
6   2006  179682690.0
7   2007  184493231.0
8   2008  189499113.0
9   2009  194376534.0
10  2010  199239047.0
11  2011  203746065.0
12  2012  207667125.0
13  2013  211073978.0
14  2014  214264647.0
15  2015  217290883.0
16  2016  220138869.0
17  2017  223273967.0
18  2018  226928892.0
19  2019  230800899.0
20  2020  235001746.0
21  2021  239477801.0
22  2022  243700667.0
23  2023  247504495.0
24  2024  251269164.0


## STEP 2: Filter and Pivot the Agricultural Data

In [3]:

# Filter for Wheat only
wheat_data = Agri_data[Agri_data['Item'] == 'Wheat']

# Pivot the data so 'Area harvested', 'Yield', and 'Production' get their own individual columns
wheat_pivoted = wheat_data.pivot(index='Year', columns='Element', values='Value').reset_index()

# Clean up column names for readability
wheat_pivoted.columns.name = None  # Remove metadata label
wheat_pivoted = wheat_pivoted.rename(columns={
    'Area harvested': 'Area_Harvested_ha',
    'Yield': 'Yield_kg_ha',
    'Production': 'Production_tonnes'
})
print(wheat_pivoted)

    Year  Area_Harvested_ha  Production_tonnes  Yield_kg_ha
0   2000          8463000.0         21078600.0       2490.7
1   2001          8180800.0         19023700.0       2325.4
2   2002          8057500.0         18226500.0       2262.1
3   2003          8033900.0         19183300.0       2387.8
4   2004          8216200.0         19499800.0       2373.3
5   2005          8358000.0         21612300.0       2585.8
6   2006          8447900.0         21276900.0       2518.6
7   2007          8578000.0         23294600.0       2715.6
8   2008          8549800.0         20958800.0       2451.4
9   2009          9046000.0         24033000.0       2656.8
10  2010          9131600.0         23310800.0       2552.8
11  2011          8900700.0         25213800.0       2832.8
12  2012          8649800.0         23473400.0       2713.8
13  2013          8660077.0         24211360.0       2795.7
14  2014          9199318.0         25979399.0       2824.1
15  2015          9203874.0         2508

## STEP 3: Combine Both Datasets

In [4]:

# Merge the vertical population data with our pivoted wheat metrics on 'Year'
final_df = pd.merge(wheat_pivoted, pak_pop_vertical, on='Year')

# Display the final structured dataset
final_df.head()

,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_ha,Population
0,2000,8463000.0,21078600.0,2490.7,154879127.0
1,2001,8180800.0,19023700.0,2325.4,159270907.0
2,2002,8057500.0,18226500.0,2262.1,163222549.0
3,2003,8033900.0,19183300.0,2387.8,167110248.0
4,2004,8216200.0,19499800.0,2373.3,171286000.0
